# Comparison: deepUMIcaller vs DupCaller

This notebook compares SNVs and INDELs called by **deepUMIcaller** and **DupCaller** at the
sample level. Both callers are run through the same deepCSA pipeline to ensure a fair
comparison.

## Outline

| Section | Description |
|---|---|
| **1. Setup** | Imports, cohort configuration, and output paths |
| **2. Data loading** | Read deepUMI, DupCaller, and combined mutation tables |
| **3. Helper functions** | Variant merging, status labelling, and data utilities |
| **4. Plotting functions** | Venn diagrams, boxplots, filter bars, heatmaps, mutation-type bars, and SBS-96 profiles |
| **5. Per-sample analysis** | Run all comparison plots and export tables for each sample |
| **6. Aggregated analysis** | Same comparison plots pooled across all samples |
| **7. Mutational profiles** | SBS-96 trinucleotide profiles per sample and aggregated |

In [2]:
# --- 1. Setup: imports, cohort configuration, output paths ---

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from matplotlib.patches import Rectangle
from matplotlib_venn import venn2

In [3]:
# --- Analysis settings --------------------------------------------------------
cohort = "bladder"          # "cord_blood" or "bladder"
mutations = "indel"  # "snv" or "all_mutation_types"

# Base directory for all outputs (plots, tables)
OUTPUT_BASE = Path(
    "./results"
)


In [4]:

# --- Cohort-specific paths --------------------------------------------------------
if cohort == "cord_blood":
    deepcsa_path = "/data/bbg/nobackup2/prominent/duplex_seq_tests/error_rate/cord_blood/bbg/deepCSA/2026-03-20_idt_tws_dupcaller_fix/mutations"
    samples = [
        # "SC001_B1_1_H_1", "SC002_B1_1_H_1", "SC003_B1_1_H_1",
        "SC001_B1_1_H_2", "SC002_B1_1_H_2", "SC003_B1_1_H_2",
    ]
    dupcaller_samples = [ f"{sample}_dupcaller" for sample in samples ]
    dupcaller_path = "/data/bbg/nobackup2/prominent/duplex_seq_tests/error_rate/cord_blood/bbg/2026-03-dupcaller/deepCSA_input/vcf/all_samples_info.maf"

elif cohort == "bladder":
    deepcsa_path = "/data/bbg/nobackup2/scratch/mhuertas/tests_dupcaller/20260318_bladder_same/mutations"
    samples = [
        "P19_0050_BDO_01", "P19_0050_BTR_01",
        "P19_0051_BDO_01", "P19_0051_BTR_01",
        "P19_0052_BDO_01", "P19_0052_BTR_01",
        "P19_0053_BDO_01", "P19_0053_BTR_01",
    ]
    dupcaller_samples = [ f"{sample}_dupcaller" for sample in samples ]
    dupcaller_path = "/data/bbg/nobackup2/scratch/fcalvet/tests_dupcaller/2026-03-11_bladder_test_035/deepCSA_input/vcf/all_samples_info.maf"

# Derived paths (shared structure for both cohorts)
all_both = f"{deepcsa_path}/germline_somatic/all_samples.filtered.tsv.gz"
all_deepumi = f"{deepcsa_path}/germline_somatic/CallerDeepumicaller.filtered.tsv.gz"
deepumi_clean = f"{deepcsa_path}/clean_somatic/CallerDeepumicaller.somatic.mutations.tsv"

print(f"Cohort: {cohort}  |  Mutation filter: {mutations}  |  Samples: {len(samples)}")

Cohort: bladder  |  Mutation filter: indel  |  Samples: 8


## 2. Data loading

Read the three input tables:

| Table | Source | Purpose |
|---|---|---|
| `dupcaller_df` | DupCaller MAF | All DupCaller calls with FILTER, LR, INFO, etc. |
| `deepumi_df` | deepCSA germline+somatic TSV | All deepUMIcaller calls (pre-filter) |
| `deepumi_clean_df` | deepCSA clean somatic TSV | Only PASS deepUMI variants (used to flag `is_pass`) |
| `mutations_table` | Combined germline+somatic gzip | Both callers together — needed for trinucleotide context |

In [5]:
def add_variant_id(df: pd.DataFrame) -> pd.DataFrame:
    """Create a unique variant identifier as CHROM:POS_REF>ALT for merging across callers."""
    df["VARIANT_ID"] = (
        df["CHROM"].astype(str) + ":"
        + df["POS"].astype(str) + "_"
        + df["REF"].astype(str) + ">"
        + df["ALT"].astype(str)
    )
    return df


# --- Read raw tables --------------------------------------------------------
dupcaller_df = pd.read_csv(dupcaller_path, sep="\t", low_memory=False)
deepumi_df = pd.read_csv(all_deepumi, sep="\t", low_memory=False)
deepumi_clean_df = pd.read_csv(deepumi_clean, sep="\t", low_memory=False)
mutations_table = pd.read_csv(all_both, sep="\t", low_memory=False, compression="gzip")

# --- Restrict to specified samples --------------------------------------------------------
dupcaller_df = dupcaller_df[dupcaller_df["SAMPLE_ID"].isin(dupcaller_samples)]
deepumi_df = deepumi_df[deepumi_df["SAMPLE_ID"].isin(samples)]
deepumi_clean_df = deepumi_clean_df[deepumi_clean_df["SAMPLE_ID"].isin(samples)]
mutations_table = mutations_table[(mutations_table["SAMPLE_ID"].isin(samples)) | (mutations_table["SAMPLE_ID"].isin(dupcaller_samples))]

# --- Add VARIANT_ID to both caller tables -----------------------------------
dupcaller_df = add_variant_id(dupcaller_df)
deepumi_df = add_variant_id(deepumi_df)

# --- Mark PASS status for deepUMI using the clean somatic set ---------------
deepumi_df["is_pass"] = deepumi_df["VARIANT_ID"].isin(deepumi_clean_df["MUT_ID"])

# --- Harmonise mutation types (deepUMI reports INSERTION/DELETION separately) -
deepumi_df["TYPE"] = deepumi_df["TYPE"].replace({"INSERTION": "INDEL", "DELETION": "INDEL"})

# --- Optional: restrict to SNVs only ----------------------------------------
if mutations == "snv":
    deepumi_df = deepumi_df[deepumi_df["TYPE"] == "SNV"]
    dupcaller_df = dupcaller_df[dupcaller_df["TYPE"] == "SNV"]

elif mutations == "indel":
    deepumi_df = deepumi_df[deepumi_df["TYPE"] == "INDEL"]
    dupcaller_df = dupcaller_df[dupcaller_df["TYPE"] == "INDEL"]

print(f"deepUMI variants: {len(deepumi_df):,}  |  DupCaller variants: {len(dupcaller_df):,}")
print(f"Mutations table rows: {len(mutations_table):,}")

deepUMI variants: 4,560  |  DupCaller variants: 2,883
Mutations table rows: 26,197


In [6]:
dupcaller_df

,CHROM,POS,REF,ALT,FILTER,INFO,FORMAT,TUMOR,DEPTH,ALT_DEPTH,LR,TYPE,SAMPLE_ID,VARIANT_ID
5,chr1,26771259,TCC,T,PASS,"F1R2=7;F2R1=6;LR=15.952056131906716;TC=7,0;BC=...",AC:RC:DP,2:7342:7345,16283,1,15.952056,INDEL,P19_0050_BDO_01_dupcaller,chr1:26771259_TCC>T
18,chrX,45085938,TC,T,PASS,"F1R2=7;F2R1=8;LR=15.85667843678884;TC=7,0;BC=8...",AC:RC:DP,2:6859:6861,7622,1,15.856678,INDEL,P19_0050_BDO_01_dupcaller,chrX:45085938_TC>T
29,chr1,26780772,T,TC,nanoseq_noise,"F1R2=6;F2R1=6;LR=15.782112271590547;TC=6,0;BC=...",AC:RC:DP,1772:5359:7299,0,2424,15.782112,INDEL,P19_0050_BDO_01_dupcaller,chr1:26780772_T>TC
30,chr1,26762167,A,AC,PASS,"F1R2=8;F2R1=6;LR=15.777082705630399;TC=8,0;BC=...",AC:RC:DP,1:7248:7260,15671,2,15.777083,INDEL,P19_0050_BDO_01_dupcaller,chr1:26762167_A>AC
31,chr1,26780776,C,CA,PASS,"F1R2=12;F2R1=6;LR=15.774979744475747;TC=12,0;B...",AC:RC:DP,6:7298:7305,12195,1,15.774980,INDEL,P19_0050_BDO_01_dupcaller,chr1:26780776_C>CA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13023,chrX,124031368,A,AT,nanoseq_noise,"F1R2=1;F2R1=1;LR=0.5056956537442492;TC=1,0;BC=...",AC:RC:DP,58:2498:2622,0,1,0.505696,INDEL,P19_0053_BTR_01_dupcaller,chrX:124031368_A>AT
13024,chrY,13335349,G,GA,nanoseq_noise,"F1R2=1;F2R1=1;LR=0.5056956537442492;TC=1,0;BC=...",AC:RC:DP,49:451:513,0,1,0.505696,INDEL,P19_0053_BTR_01_dupcaller,chrY:13335349_G>GA
13025,chrY,13336337,T,TA,nanoseq_noise,"F1R2=1;F2R1=1;LR=0.5056956537442492;TC=1,0;BC=...",AC:RC:DP,23:85:118,0,1,0.505696,INDEL,P19_0053_BTR_01_dupcaller,chrY:13336337_T>TA
13028,chr13,48347600,C,CT,nanoseq_noise,"F1R2=3;F2R1=1;LR=0.5040600197669318;TC=2,1;BC=...",AC:RC:DP,44:5008:5148,0,1,0.504060,INDEL,P19_0053_BTR_01_dupcaller,chr13:48347600_C>CT


## 3. Helper functions

Core utilities for merging variant tables and computing overlap statistics.

In [7]:
STATUS_COLORS = {
    "Only deepUMI":   "#4E79A7",
    "Only DupCaller": "#F28E2B",
    "Common":         "#A08369",
}


def build_merged_variants(
    sample_deepumi_df: pd.DataFrame,
    sample_dupcaller_df: pd.DataFrame,
    deepumi_pass_only: bool = False,
    dupcaller_pass_only: bool = False,
) -> pd.DataFrame:
    """Outer-merge deepUMI and DupCaller variant tables and label each row.

    Parameters
    ----------
    sample_deepumi_df : DataFrame
        deepUMIcaller variants (must contain VARIANT_ID, SAMPLE_ID, FILTER, VAF,
        TYPE, DEPTH, ALT_DEPTH, and the boolean ``is_pass`` column).
    sample_dupcaller_df : DataFrame
        DupCaller variants (must contain VARIANT_ID, SAMPLE_ID, FILTER, LR,
        TYPE, DEPTH, ALT_DEPTH, INFO).
    deepumi_pass_only : bool
        If True, keep only deepUMI variants where ``is_pass == True``.
    dupcaller_pass_only : bool
        If True, keep only DupCaller variants where ``FILTER == "PASS"``.

    Returns
    -------
    DataFrame
        Merged table with a ``status`` column: "Common", "Only deepUMI", or
        "Only DupCaller".
    """
    # Apply PASS filters when requested
    if deepumi_pass_only:
        deepumi_subset = sample_deepumi_df[sample_deepumi_df["is_pass"] == True].copy()
    else:
        deepumi_subset = sample_deepumi_df.copy()

    if dupcaller_pass_only:
        dupcaller_subset = sample_dupcaller_df[sample_dupcaller_df["FILTER"] == "PASS"].copy()
    else:
        dupcaller_subset = sample_dupcaller_df.copy()

    # Select and rename columns so they don't collide after the merge
    deepumi_subset = deepumi_subset[
        ["VARIANT_ID", "SAMPLE_ID", "FILTER", "VAF", "TYPE", "DEPTH", "ALT_DEPTH"]
    ].rename(columns={
        "FILTER": "FILTER_deepumi",
        "VAF": "VAF_deepumi",
        "TYPE": "TYPE_deepumi",
        "DEPTH": "DEPTH_deepumi",
        "ALT_DEPTH": "ALT_DEPTH_deepumi",
    })

    dupcaller_subset = dupcaller_subset[
        ["SAMPLE_ID", "VARIANT_ID", "FILTER", "LR", "TYPE", "DEPTH", "ALT_DEPTH", "INFO"]
    ].rename(columns={
        "FILTER": "FILTER_dupcaller",
        "LR": "LR_dupcaller",
        "TYPE": "TYPE_dupcaller",
        "DEPTH": "DEPTH_dupcaller",
        "ALT_DEPTH": "ALT_DEPTH_dupcaller",
        "INFO": "INFO_dupcaller",
    })

    # Remove the "_dupcaller" suffix so SAMPLE_IDs match
    dupcaller_subset["SAMPLE_ID"] = dupcaller_subset["SAMPLE_ID"].str.replace(
        "_dupcaller", "", regex=False
    )

    # Outer merge with indicator
    merged_df = pd.merge(
        deepumi_subset, dupcaller_subset,
        on=["VARIANT_ID", "SAMPLE_ID"],
        how="outer",
        indicator=True,
    )

    status_map = {
        "both": "Common",
        "left_only": "Only deepUMI",
        "right_only": "Only DupCaller",
    }
    merged_df["status"] = merged_df["_merge"].map(status_map)
    return merged_df.drop(columns=["_merge"])


def status_counts(merged_df: pd.DataFrame) -> tuple[int, int, int]:
    """Return (only_deepumi, only_dupcaller, common) counts from a merged table."""
    only_deepumi = int((merged_df["status"] == "Only deepUMI").sum())
    only_dupcaller = int((merged_df["status"] == "Only DupCaller").sum())
    common = int((merged_df["status"] == "Common").sum())
    return only_deepumi, only_dupcaller, common


def _save_figure(fig: plt.Figure, *path_parts: str) -> None:
    """Save a figure under OUTPUT_BASE, creating parent directories as needed."""
    save_path = OUTPUT_BASE / Path(*path_parts)
    save_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(save_path)
    plt.close(fig)

## 4. Plotting functions

All visualisation routines. Each function receives pre-computed DataFrames and a
sample name, generates the plot, and saves it to disk.

In [8]:
def plot_venn_pair(
    merged_all_df: pd.DataFrame,
    merged_pass_both_df: pd.DataFrame,
    sample_name: str,
) -> None:
    """Side-by-side Venn diagrams: all variants vs PASS-in-both variants.

    Left panel shows the full overlap regardless of filter status; right panel
    restricts to variants that pass QC in both callers.
    """
    fig, axes = plt.subplots(1, 2, figsize=(8,4))
    panels = [
        (merged_all_df, "All variants"),
        (merged_pass_both_df, "PASS in both callers"),
    ]

    for ax, (df, title) in zip(axes, panels):
        only_deepumi, only_dupcaller, common = status_counts(df)
        venn = venn2(
            subsets=(only_deepumi, only_dupcaller, common),
            set_labels=("deepUMIcaller", "DupCaller"),
            set_colors=(STATUS_COLORS["Only deepUMI"], STATUS_COLORS["Only DupCaller"]),
            alpha=0.8,
            ax=ax,
        )
        # Colour the overlap region
        if venn.get_patch_by_id("11") is not None:
            venn.get_patch_by_id("11").set_color(STATUS_COLORS["Common"])
            venn.get_patch_by_id("11").set_alpha(0.8)
        ax.set_title(title)

    fig.suptitle(sample_name)
    fig.tight_layout(rect=[0, 0, 1, 0.95])
    _save_figure(fig, cohort, "plots", mutations, "venn", f"{sample_name}.pdf")


def plot_boxplot_pair(
    merged_all_df: pd.DataFrame,
    merged_pass_df: pd.DataFrame,
    sample_name: str,
    column: str,
    caller: str,
) -> None:
    """Box-plots comparing a numeric feature between Common and caller-unique variants.

    Parameters
    ----------
    column : str
        Column to plot (e.g. "LR_dupcaller", "VAF_deepumi").
    caller : str
        Which caller's unique variants to show ("DupCaller" or "deepUMI").
    """
    fig, axes = plt.subplots(1, 2, figsize=(6, 4))
    panels = [
        (merged_all_df, "All variants"),
        (merged_pass_df, "PASS-in-both variants"),
    ]

    for ax, (df, title) in zip(axes, panels):
        common_vals = df[df["status"] == "Common"][column].dropna()
        unique_vals = df[df["status"] == f"Only {caller}"][column].dropna()

        bp = ax.boxplot(
            [common_vals, unique_vals],
            tick_labels=["Common", f"Only {caller}"],
            patch_artist=True,
            medianprops={"color": "black", "linewidth": 2},
        )
        for patch, color in zip(
            bp["boxes"],
            [STATUS_COLORS["Common"], STATUS_COLORS[f"Only {caller}"]],
        ):
            patch.set_facecolor(color)
            patch.set_alpha(0.7)

        ax.set_ylabel(column)
        ax.set_title(title)
        ax.grid(axis="y", linestyle="--", alpha=0.7)

    fig.suptitle(f"Per-caller distribution: {sample_name}")
    fig.tight_layout(rect=[0, 0, 1, 0.95])
    _save_figure(fig, cohort, "plots", mutations, column, f"{sample_name}.png")


def plot_violinplot_pair(
    merged_all_df: pd.DataFrame,
    merged_pass_df: pd.DataFrame,
    sample_name: str,
    column: str,
    caller: str,
) -> None:
    """Violin-plots comparing a numeric feature between Common and caller-unique variants.

    Parameters
    ----------
    column : str
        Column to plot (e.g. "LR_dupcaller", "VAF_deepumi").
    caller : str
        Which caller's unique variants to show ("DupCaller" or "deepUMI").
    """
    fig, axes = plt.subplots(1, 2, figsize=(8, 5), sharey=True)
    panels = [
        (merged_all_df, "All variants"),
        (merged_pass_df, "PASS-in-both variants"),
    ]

    for ax, (df, title) in zip(axes, panels):
        common_vals = df[df["status"] == "Common"][column].dropna()
        unique_vals = df[df["status"] == f"Only {caller}"][column].dropna()
        
        data_to_plot = [common_vals if not common_vals.empty else [0], 
                        unique_vals if not unique_vals.empty else [0]]

        vparts = ax.violinplot(
            data_to_plot,
            positions=[1, 2],
            showmeans=False,
            showmedians=True,
            showextrema=False
        )
        
        colors = [STATUS_COLORS["Common"], STATUS_COLORS[f"Only {caller}"]]
        
        # Color the violin bodies
        for pc, color in zip(vparts['bodies'], colors):
            pc.set_facecolor(color)
            pc.set_edgecolor('black')
            pc.set_alpha(0.7)
            
        # Color the lines (median, extrema)
        for partname in ['cmedians']:
            vp = vparts[partname]
            vp.set_edgecolor('black')
            vp.set_linewidth(1)


        # 2. Add Stripplot
        # We use a temporary dataframe for seaborn to handle categorical mapping correctly
        strip_data = pd.DataFrame({
            "val": np.concatenate([common_vals, unique_vals]),
            "grp":  ["Common"] * len(common_vals) + [f"Only {caller}"] * len(unique_vals)
        })
        
        # if not strip_data.empty:
        #     sns.stripplot(
        #         data=strip_data,
        #         x="grp",
        #         y="val",
        #         order=["Extra", "Common", f"Only {caller}"],
        #         jitter=True,
        #         color='black',
        #         alpha=0.4,
        #         ax=ax,
        #         size=3,
        #         zorder=2
        #     )


        ax.set_xticks([1, 2])
        ax.set_xticklabels(["Common", f"Only {caller}"])
        ax.set_ylabel(column)
        ax.set_title(title)
        ax.grid(axis="y", linestyle="--", alpha=0.3)

    fig.suptitle(f"Per-caller distribution: {sample_name} ({column})")
    fig.tight_layout(rect=[0, 0, 1, 0.95])
    _save_figure(fig, cohort, "plots", mutations, column, f"violin_{sample_name}.pdf")


def plot_deepumi_filter_for_dupcaller_pass(
    merged_all_df: pd.DataFrame,
    merged_pass_df: pd.DataFrame,
    sample_name: str,
    top_n: int = 10,
) -> None:
    """Bar chart of deepUMI FILTER reasons for variants that PASS in DupCaller.

    Identifies common variants (present in both callers in the unfiltered merge)
    that are *not* PASS in deepUMI but *are* PASS in DupCaller, then shows the
    most frequent deepUMI filter labels.
    """
    # Variants that already pass in deepUMI (or are deepUMI-unique) — exclude them
    deepumi_pass_ids = merged_pass_df[
        merged_pass_df["status"].isin(["Common", "Only deepUMI"])
    ]["VARIANT_ID"]

    # Common variants in the unfiltered merge that did NOT pass deepUMI
    non_pass_common = merged_all_df[
        (~merged_all_df["VARIANT_ID"].isin(deepumi_pass_ids))
        & (merged_all_df["status"] == "Common")
    ]

    filter_counts = (
        non_pass_common["FILTER_deepumi"]
        .fillna("NA").astype(str)
        .value_counts()
        .sort_values(ascending=False)
        .head(top_n)
    )

    fig, ax = plt.subplots(figsize=(12, 8))

    if filter_counts.empty:
        ax.text(0.5, 0.5, "No Common variants with DupCaller PASS",
                ha="center", va="center", fontsize=11, transform=ax.transAxes)
        ax.axis("off")
    else:
        filter_counts.plot(kind="bar", color=[STATUS_COLORS["Common"]], ax=ax)
        for i, v in enumerate(filter_counts.values):
            ax.text(i, v, str(v), ha="center", va="bottom")
        ax.set_ylabel("Number of variants")
        ax.set_xlabel("deepUMI FILTER")
        ax.set_xticklabels(filter_counts.index, rotation=55, ha="right")
        ax.grid(axis="y", linestyle="--", alpha=0.5)

    ax.set_title(f"Top {top_n} deepUMI filters for DupCaller PASS variants: {sample_name}")
    fig.subplots_adjust(left=0.12, right=0.98, bottom=0.60, top=0.90)
    _save_figure(
        fig, cohort, "plots", mutations, "filters",
        f"deepumi_filter_for_dupcaller_pass_{sample_name}.png",
    )


def plot_dupcaller_family_heatmap(
    dupcaller_unique_df: pd.DataFrame,
    sample_name: str,
) -> None:
    """Heatmap of F1R2 × F2R1 family-size combinations for DupCaller-unique variants.

    Also shows a stacked bar summarising how many mutations have minimal support
    (family size 2; 1+1) vs higher support (>2; >1+>1).
    """
    df = dupcaller_unique_df.copy()

    # Extract F1R2 and F2R1 counts from the INFO field
    df["F1R2"] = pd.to_numeric(
        df["INFO_dupcaller"].str.extract(r'(?:^|;)F1R2="?([^;"]+)"?')[0],
        errors="coerce",
    )
    df["F2R1"] = pd.to_numeric(
        df["INFO_dupcaller"].str.extract(r'(?:^|;)F2R1="?([^;"]+)"?')[0],
        errors="coerce",
    )

    # Keep only rows with at least 1 read on each strand
    binned_df = (
        df.dropna(subset=["F1R2", "F2R1"])
        .assign(F1R2=lambda x: x["F1R2"].astype(int),
                F2R1=lambda x: x["F2R1"].astype(int))
        .query("F1R2 >= 1 and F2R1 >= 1")
    )

    bin_labels = ["1", "2", "3", "4", "5+"]

    if binned_df.empty:
        heatmap_df = pd.DataFrame(0, index=bin_labels, columns=bin_labels)
        counts = pd.Series({"Using SSC size 1": 0, "Both SSC bigger than 1": 0})
    else:
        # Bin values: 1–4 individually, ≥5 grouped
        binned_df["F1R2_bin"] = np.where(binned_df["F1R2"] >= 5, "5+", binned_df["F1R2"].astype(str))
        binned_df["F2R1_bin"] = np.where(binned_df["F2R1"] >= 5, "5+", binned_df["F2R1"].astype(str))

        combo_counts = (
            binned_df.groupby(["F1R2_bin", "F2R1_bin"])
            .size().reset_index(name="n_mutations")
        )
        heatmap_df = (
            combo_counts.pivot(index="F1R2_bin", columns="F2R1_bin", values="n_mutations")
            .reindex(index=bin_labels, columns=bin_labels, fill_value=0)
            .fillna(0).astype(int)
        )

        # Categorise: minimal support (either strand has only 1 read) vs higher
        binned_df["category"] = np.where(
            (binned_df["F1R2"] == 1) | (binned_df["F2R1"] == 1),
            "Using SSC size 1", "Both SSC bigger than 1",
        )
        counts = binned_df["category"].value_counts().reindex(
            ["Using SSC size 1", "Both SSC bigger than 1"], fill_value=0
        )

    # --- Plot: heatmap + stacked bar -----------------------------------------
    fig, (ax1, ax2) = plt.subplots(
        1, 2, figsize=(7, 5), gridspec_kw={"width_ratios": [3, 1]}
    )

    sns.heatmap(heatmap_df, cmap="viridis", annot=True, fmt="d",
                cbar_kws={"label": "# mutations"}, ax=ax1)
    ax1.set_title(f"DupCaller unique mutation counts\nby each combination of reads\nsupporting each SSC family:\n{sample_name}")
    ax1.set_xlabel("SSC size 1")
    ax1.set_ylabel("SSC size 2")

    # Stacked bar (viridis endpoints for consistency)
    color_low, color_high = "#fde725", "#440154"
    ax2.bar([sample_name], counts["Using SSC size 1"],
            label="Using SSC size 1", color=color_low)
    ax2.bar([sample_name], counts["Both SSC bigger than 1"],
            bottom=counts["Using SSC size 1"],
            label="Both SSC bigger than 1", color=color_high)

    if counts["Using SSC size 1"] > 0:
        ax2.text(0, counts["Using SSC size 1"] / 2,
                 str(int(counts["Using SSC size 1"])),
                 ha="center", va="center", color="black", fontweight="bold")
    if counts["Both SSC bigger than 1"] > 0:
        ax2.text(0, counts["Using SSC size 1"] + counts["Both SSC bigger than 1"] / 2,
                 str(int(counts["Both SSC bigger than 1"])),
                 ha="center", va="center", color="white", fontweight="bold")

    ax2.set_ylabel("Number of mutations")
    ax2.set_title("Single strand\nfamilies' support")
    ax2.legend()

    plt.tight_layout()
    _save_figure(
        fig, cohort, "plots", mutations, "dupcaller_unique",
        f"reads_{sample_name}.pdf",
    )


def plot_mutation_types(
    merged_all_df: pd.DataFrame,
    merged_pass_both_df: pd.DataFrame,
    sample_name: str,
    mutations: str,
) -> None:
    """Grouped bar chart comparing SNV / INDEL counts by caller status.

    Shows two panels: all variants (no PASS filter) and PASS-only variants.
    Note: for Common variants the type is taken from deepUMI; for DupCaller-
    unique variants it is taken from DupCaller.
    """
    mutations_list = ["SNV"] if mutations == "snv" else ["SNV", "INDEL"]

    def _prepare_counts(df: pd.DataFrame, panel: str) -> pd.DataFrame:
        x = df.copy()
        x["mutation_type"] = np.where(
            x["status"] == "Only DupCaller", x["TYPE_dupcaller"], x["TYPE_deepumi"]
        )
        x = x[x["mutation_type"].isin(mutations_list)]
        out = (x.groupby(["mutation_type", "status"], observed=False)
                .size().reset_index(name="n_mutations"))
        out["panel"] = panel
        return out

    plot_df = pd.concat([
        _prepare_counts(merged_all_df, "All (no PASS filter)"),
        _prepare_counts(merged_pass_both_df, "PASS only"),
    ], ignore_index=True)

    # Ensure all combinations exist (including zero counts)
    panel_order = ["All (no PASS filter)", "PASS only"]
    status_order = ["Common", "Only deepUMI", "Only DupCaller"]
    full_idx = pd.MultiIndex.from_product(
        [panel_order, mutations_list, status_order],
        names=["panel", "mutation_type", "status"],
    )
    plot_df = (
        plot_df.set_index(["panel", "mutation_type", "status"])
        .reindex(full_idx, fill_value=0).reset_index()
    )

    fig, axes = plt.subplots(1, 2, figsize=(15, 5), sharey=True)
    for ax, panel in zip(axes, panel_order):
        d = plot_df[plot_df["panel"] == panel]
        sns.barplot(data=d, x="mutation_type", y="n_mutations",
                    hue="status", hue_order=status_order,
                    palette=STATUS_COLORS, ax=ax)
        ax.set_title(panel)
        ax.set_xlabel("Mutation type")
        ax.set_ylabel("Number of mutations")
        ax.grid(axis="y", linestyle="--", alpha=0.4)

        for container in ax.containers:
            labels = [f"{int(v)}" if pd.notna(v) else "" for v in container.datavalues]
            ax.bar_label(container, labels=labels, padding=2, fontsize=9)

    fig.suptitle("Mutation types", y=1.02)
    fig.tight_layout()
    _save_figure(
        fig, cohort, "plots", mutations, "mutation_types",
        f"{sample_name.replace(' ', '_')}.pdf",
    )

## 5. Per-sample analysis

For each sample we:
1. Build the merged variant table (all variants **and** PASS-only).
2. Generate Venn diagrams, box-plots, filter-reason bars, family-size heatmaps,
   and mutation-type bar charts.
3. Export the full merged table as a TSV.

In [9]:
# top_n_deepumi_filters = 10

# for sample in samples:
#     print(f"Processing {sample} …")

#     # --- Subset per sample ---------------------------------------------------
#     sample_deepumi_df = deepumi_df[deepumi_df["SAMPLE_ID"] == sample]
#     sample_deepumi_pass_df = deepumi_clean_df[deepumi_clean_df["SAMPLE_ID"] == sample]
#     sample_dupcaller_df = dupcaller_df[dupcaller_df["SAMPLE_ID"] == f"{sample}_dupcaller"]

#     # --- Merge: all variants & PASS-only -------------------------------------
#     merged_all_df = build_merged_variants(sample_deepumi_df, sample_dupcaller_df)
#     merged_pass_both_df = build_merged_variants(
#         sample_deepumi_df, sample_dupcaller_df,
#         deepumi_pass_only=True, dupcaller_pass_only=True,
#     )

#     # DupCaller-unique variants (PASS filter applied)
#     dupcaller_unique = merged_pass_both_df[
#         merged_pass_both_df["status"] == "Only DupCaller"
#     ].copy()

#     # --- Generate all plots --------------------------------------------------
#     plot_venn_pair(merged_all_df, merged_pass_both_df, sample)
#     plot_boxplot_pair(merged_all_df, merged_pass_both_df, sample, "LR_dupcaller", "DupCaller")
#     plot_violinplot_pair(merged_all_df, merged_pass_both_df, sample, "LR_dupcaller", "DupCaller")
#     plot_boxplot_pair(merged_all_df, merged_pass_both_df, sample, "VAF_deepumi", "deepUMI")
#     plot_deepumi_filter_for_dupcaller_pass(
#         merged_all_df, merged_pass_both_df, sample, top_n=top_n_deepumi_filters,
#     )
#     plot_dupcaller_family_heatmap(dupcaller_unique, sample)
#     plot_mutation_types(merged_all_df, merged_pass_both_df, sample, mutations)

#     # --- Export merged table -------------------------------------------------
#     output_dir = OUTPUT_BASE / cohort / "mutation_comparison" / mutations
#     output_dir.mkdir(parents=True, exist_ok=True)
#     merged_all_df.sort_values(by="status").to_csv(
#         output_dir / f"{sample}.tsv", index=False, sep="\t",
#     )


## 6. Aggregated analysis (all samples pooled)

Same plots as above but merging all samples together to get a cohort-level view.

In [10]:
# Merge across all samples at once
merged_all_df = build_merged_variants(deepumi_df, dupcaller_df)
merged_pass_both_df = build_merged_variants(
    deepumi_df, dupcaller_df,
    deepumi_pass_only=True, dupcaller_pass_only=True,
)

dupcaller_unique_df = merged_all_df[merged_all_df["status"] == "Only DupCaller"].copy()

# Generate cohort-level plots
plot_venn_pair(merged_all_df, merged_pass_both_df, f"All_{cohort}_samples")
plot_mutation_types(merged_all_df, merged_pass_both_df, "All samples", mutations)
plot_dupcaller_family_heatmap(dupcaller_unique_df, "All samples")
# plot_boxplot_pair(merged_all_df, merged_pass_both_df, "All samples", "LR_dupcaller", "DupCaller")
plot_violinplot_pair(merged_all_df, merged_pass_both_df, "All samples", "LR_dupcaller", "DupCaller")

## 7. Mutational profiles (SBS-96)

Trinucleotide-context mutational spectra following the standard COSMIC SBS-96
representation. Each bar corresponds to one of the 96 possible single-base
substitutions in their trinucleotide context, grouped by the six pyrimidine
substitution classes (C>A, C>G, C>T, T>A, T>C, T>G).

Profiles are generated separately for:
- **Only deepUMI** – variants exclusive to deepUMIcaller
- **Only DupCaller** – variants exclusive to DupCaller
- **Common** – variants found by both callers

In [11]:
# Standard SBS-96 colour palette
SBS_COLORS = {
    "C>A": "#03bcee",
    "C>G": "#010101",
    "C>T": "#e32926",
    "T>A": "#cac9c9",
    "T>C": "#a1ce63",
    "T>G": "#ebc6c4",
}

# Canonical substitution classes and trinucleotide ordering
SBS_CLASSES = ["C>A", "C>G", "C>T", "T>A", "T>C", "T>G"]
NUCLEOTIDES = ["A", "C", "G", "T"]


def plot_sbs96_profile(
    frequencies: dict[str, float],
    title: str = "Mutational profile",
    yaxis_name: str = "% SBS",
    output_f: str | Path | None = None,
) -> None:
    """Plot an SBS-96 trinucleotide mutational profile.

    Parameters
    ----------
    frequencies : dict
        Mapping from trinucleotide change (e.g. "ACG>T") to its normalised
        frequency.
    title : str
        Plot title (typically includes sample name and mutation count).
    yaxis_name : str
        Label for the y-axis.
    output_f : str or Path, optional
        If given, save the figure to this path at 600 dpi.
    """
    fig, ax = plt.subplots(
        2, sharex="col", figsize=(6, 1.5),
        gridspec_kw={"height_ratios": [0.05, 0.95]},
    )
    plt.subplots_adjust(left=0.1, bottom=0.1, right=0.9, top=0.9,
                        wspace=0.01, hspace=0)
    plt.title(title, fontsize=7, y=1.2)

    # --- Top axis: coloured strips labelling the six substitution classes ------
    ax_top = ax[0]
    for spine in ax_top.spines.values():
        spine.set_visible(False)
    ax_top.set_yticks([])

    fontsize_label = 5
    strip_width = 15.75
    for i, sbs in enumerate(SBS_CLASSES):
        x_offset = i * 16
        ax_top.add_patch(
            Rectangle((x_offset, 0.15 if i == 0 else 0.1), strip_width, 0.5,
                       color=SBS_COLORS[sbs])
        )
        ax_top.text(x_offset + strip_width / 2 - (0 if i == 0 else 1), 0.75,
                    sbs, fontsize=fontsize_label, weight="normal", ha="center")

    ax_bar = ax[1]
    probabilities, colors, labels = [], [], []
    for sbs in SBS_CLASSES:
        ref, alt = sbs.split(">")
        for nuc5 in NUCLEOTIDES:
            for nuc3 in NUCLEOTIDES:
                trinuc = f"{nuc5}{ref}{nuc3}"
                probabilities.append(frequencies.get(f"{trinuc}>{alt}", 0))
                colors.append(SBS_COLORS[sbs])
                labels.append(trinuc)

    ax_bar.bar(range(96), probabilities, width=0.8, color=colors)
    ax_bar.set_xlim(-1, 96)
    ax_bar.set_xticks(range(96))
    ax_bar.set_xticklabels(labels, fontsize=3.5, color="black", va="top", ha="center")
    plt.xticks(rotation=90)
    ax_bar.set_ylabel(yaxis_name, fontsize=6)
    ax_bar.set_yticklabels(ax_bar.get_yticklabels(), fontsize=3.5, color="black",
                           va="center", ha="center")
    for a in ax:
        a.set_axisbelow(True)
        for tick in a.yaxis.get_major_ticks():
            tick.label1.set_fontsize(3.5)
        for spine in a.spines.values():
            spine.set_linewidth(0.25)
        a.tick_params(axis="x", which="both", bottom=False)
        a.tick_params(axis="y", which="both", left=False)
        a.tick_params(axis="y", which="major", pad=2)
        a.tick_params(axis="x", which="major", pad=0)

    if output_f:
        output_f = Path(output_f)
        output_f.parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(output_f, bbox_inches="tight", dpi=600)
        plt.close(fig)

In [12]:
# # --- Per-sample SBS-96 profiles (PASS variants only) -------------------------
# # For each sample and each status (Common / Only deepUMI / Only DupCaller),
# # retrieve the trinucleotide context from the combined mutations table and plot

# mutations_table_snvs = mutations_table[mutations_table["TYPE"] == "SNV"].copy()

# for sample in samples:
#     sample_deepumi_df = deepumi_df[deepumi_df["SAMPLE_ID"] == sample]
#     sample_dupcaller_df = dupcaller_df[dupcaller_df["SAMPLE_ID"] == f"{sample}_dupcaller"]

#     merged_pass_both_df = build_merged_variants(
#         sample_deepumi_df, sample_dupcaller_df,
#         deepumi_pass_only=True, dupcaller_pass_only=True,
#     )

#     for status in ["Only deepUMI", "Only DupCaller", "Common"]:
#         subset_ids = merged_pass_both_df[
#             merged_pass_both_df["status"] == status
#         ]["VARIANT_ID"]

#         # Match the correct SAMPLE_ID in the mutations table
#         if status == "Only DupCaller":
#             sample_muts = mutations_table_snvs[
#                 (mutations_table_snvs["SAMPLE_ID"] == f"{sample}_dupcaller")
#                 & mutations_table_snvs["MUT_ID"].isin(subset_ids)
#             ]
#         elif status == "Only deepUMI":
#             sample_muts = mutations_table_snvs[
#                 (mutations_table_snvs["SAMPLE_ID"] == sample)
#                 & mutations_table_snvs["MUT_ID"].isin(subset_ids)
#             ]
#         else:
#             # Common variants appear under both caller SAMPLE_IDs — deduplicate
#             sample_muts = mutations_table_snvs[
#                 mutations_table_snvs["SAMPLE_ID"].isin([sample, f"{sample}_dupcaller"])
#                 & mutations_table_snvs["MUT_ID"].isin(subset_ids)
#             ].drop_duplicates(subset=["MUT_ID"])

#         # Count trinucleotide contexts and normalise
#         context_counts = sample_muts.groupby("CONTEXT_MUT").size().to_frame(sample)
#         total = context_counts[sample].sum()
#         context_counts[sample] = context_counts[sample] / total

#         status_slug = status.lower().replace(" ", "_")
#         plot_sbs96_profile(
#             dict(zip(context_counts.index, context_counts[sample])),
#             title=f"{sample} ({round(total)} muts)",
#             output_f=(
#                 OUTPUT_BASE / cohort / "plots" / "profiles"
#                 / sample.replace("_dupcaller", "") / f"{status_slug}.pdf"
#             ),
#         )

In [13]:
# --- Aggregated SBS-96 profiles (all samples pooled) -------------------------
# Uses an inner merge on (MUT_ID, SAMPLE_BASE) to correctly match variants to
# their status without double-counting Common variants.

merged_pass_both_df = build_merged_variants(
    deepumi_df, dupcaller_df,
    deepumi_pass_only=True, dupcaller_pass_only=True,
)

mutations_table_snvs = mutations_table[mutations_table["TYPE"] == "SNV"].copy()
mutations_table_snvs["SAMPLE_BASE"] = mutations_table_snvs["SAMPLE_ID"].str.replace(
    "_dupcaller", "", regex=False
)

sample_label = "All samples"

for status in ["Only DupCaller", "Only deepUMI", "Common"]:
    # Get the (Variant, Sample) pairs for this status — the source of truth
    relevant_pairs = merged_pass_both_df[
        merged_pass_both_df["status"] == status
    ][["VARIANT_ID", "SAMPLE_ID"]]

    # Inner merge to filter the mutations table to exactly these pairs
    sample_muts = mutations_table_snvs.merge(
        relevant_pairs,
        left_on=["MUT_ID", "SAMPLE_BASE"],
        right_on=["VARIANT_ID", "SAMPLE_ID"],
        how="inner",
    )

    # Deduplicate Common variants (appear once per caller in the mutations table)
    sample_muts = sample_muts.drop_duplicates(subset=["MUT_ID", "SAMPLE_BASE"])

    if sample_muts.empty:
        continue

    context_counts = sample_muts.groupby("CONTEXT_MUT").size().to_frame(sample_label)
    total = int(context_counts[sample_label].sum())
    context_counts[sample_label] = context_counts[sample_label] / total

    status_slug = status.lower().replace(" ", "_")
    plot_sbs96_profile(
        dict(zip(context_counts.index, context_counts[sample_label])),
        title=f"{sample_label} - {status} ({total} muts)",
        output_f=(
            OUTPUT_BASE / cohort / "plots" / "profiles" / "all_samples"
            / f"{status_slug}.pdf"
        ),
    )